# 03 SSM Forecast

本研究の主分析モデルである local linear trend + seasonal + fixed exog の状態空間モデルを、既存の予測評価基盤に接続する。推定は log scale の `y` で行い、予測後に `exp()` で `number_parcels` の原系列スケールへ戻して評価する。

今回は fixed A/B の conditional forecast のみを扱う。外生変数は `baseline_m4` 仕様を使い、学習期間で定数の列は自動除外する。

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "Transport_amount_project" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_connected_parcel_data
from src.forecasting.evaluation import evaluate_forecasts
from src.forecasting.splits import make_fixed_split_a, make_fixed_split_b
from src.forecasting.ssm import (
    fit_ssm,
    forecast_ssm,
    prepare_ssm_exog_from_spec,
    select_nonconstant_exog,
)


DATA_PATH = PROJECT_ROOT / "data" / "processed" / "parcel_volume_connected.csv"
FORECAST_DIR = PROJECT_ROOT / "output" / "forecasts"
PREDICTIONS_DIR = FORECAST_DIR / "predictions"
METRICS_DIR = FORECAST_DIR / "metrics"
FIGURES_DIR = FORECAST_DIR / "figures"

for path in [PREDICTIONS_DIR, METRICS_DIR, FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

SSM_SPEC = "baseline_m4"

print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)
print("Forecast output:", FORECAST_DIR)

## 1. データ読み込みと固定分割

接続済みデータを読み込み、fixed A/B を作成する。fixed B がデータ期間不足で作れない環境でも notebook が止まらないよう、失敗時は理由を表示して skip する。

In [ ]:
df = load_connected_parcel_data(str(DATA_PATH))

splits = {}
split_rows = []
for split_name, maker in [("fixed_a", make_fixed_split_a), ("fixed_b", make_fixed_split_b)]:
    try:
        split = maker(df)
        splits[split_name] = split
        split_rows.append(
            {
                "split": split_name,
                "status": "success",
                "train_start": split["train"].index.min().date(),
                "train_end": split["train"].index.max().date(),
                "train_rows": len(split["train"]),
                "test_start": split["test"].index.min().date(),
                "test_end": split["test"].index.max().date(),
                "test_rows": len(split["test"]),
                "message": "",
            }
        )
    except ValueError as exc:
        split_rows.append(
            {
                "split": split_name,
                "status": "skipped",
                "train_start": None,
                "train_end": None,
                "train_rows": 0,
                "test_start": None,
                "test_end": None,
                "test_rows": 0,
                "message": str(exc),
            }
        )

split_summary = pd.DataFrame(split_rows)
display(split_summary)

## 2. SSM conditional forecast

`baseline_m4` のイベントダミーを train/test 全体で作成し、学習期間で変動しない列を除外する。fixed A では COVID 系ダミーと `post_stat_change` が学習期間で全て0になるため、主に `hike_dummy` だけが残る。

In [ ]:
prediction_frames = []
fit_rows = []
used_exog_by_split = {}

for split_name, split in splits.items():
    train = split["train"]
    test = split["test"]
    print(f"Fitting {split_name} SSM")

    exog_all = prepare_ssm_exog_from_spec(pd.concat([train, test]), SSM_SPEC)
    exog_train = exog_all.loc[train.index]
    exog_test = exog_all.loc[test.index]
    exog_train_selected, exog_test_selected, used_exog = select_nonconstant_exog(exog_train, exog_test)
    used_exog_by_split[split_name] = used_exog

    result = fit_ssm(train, exog_train_selected)
    forecast = forecast_ssm(result, test, exog_test_selected, split=split_name, spec_name=SSM_SPEC)
    prediction_frames.append(forecast)

    retvals = getattr(result, "mle_retvals", {}) or {}
    fit_rows.append(
        {
            "model": "ssm",
            "split": split_name,
            "forecast_type": "conditional",
            "spec_name": SSM_SPEC,
            "used_exog": ",".join(used_exog),
            "converged": retvals.get("converged"),
            "warnflag": retvals.get("warnflag"),
            "iterations": retvals.get("iterations"),
            "log_likelihood": float(result.llf),
            "aic": float(result.aic),
            "bic": float(result.bic),
        }
    )

predictions_df = pd.concat(prediction_frames, ignore_index=True)
fit_summary = pd.DataFrame(fit_rows)

display(fit_summary)
display(pd.DataFrame([{"split": k, "used_exog": ",".join(v)} for k, v in used_exog_by_split.items()]))

## 3. 評価指標と既存baselineとの比較

SSM の予測を `number_parcels` スケールで評価する。既存の naive metrics / SARIMAX metrics がある場合は読み込み、同じ表で比較する。

In [ ]:
metrics_frames = []
for split_name, split in splits.items():
    split_predictions = predictions_df[predictions_df["split"] == split_name]
    split_metrics = evaluate_forecasts(
        split_predictions,
        y_train=split["train"]["number_parcels"],
    )
    metrics_frames.append(split_metrics)

ssm_metrics = pd.concat(metrics_frames, ignore_index=True)

comparison_frames = []
for metrics_name in ["naive_metrics.csv", "sarimax_metrics.csv"]:
    metrics_path = METRICS_DIR / metrics_name
    if metrics_path.exists():
        comparison_frames.append(pd.read_csv(metrics_path))
comparison_frames.append(ssm_metrics)
comparison_metrics = pd.concat(comparison_frames, ignore_index=True)

display(ssm_metrics)
display(comparison_metrics.sort_values(["split", "rmse"]))

## 4. 予測結果と評価指標の保存

共通フォーマットの予測結果、評価指標、fit summary を `output/forecasts/` 配下に保存する。論文用の `output/tables/` と `output/figures/` は使わない。

In [ ]:
if "fixed_a" in splits:
    predictions_df[predictions_df["split"] == "fixed_a"].to_csv(PREDICTIONS_DIR / "fixed_a_ssm.csv", index=False)

if "fixed_b" in splits:
    predictions_df[predictions_df["split"] == "fixed_b"].to_csv(PREDICTIONS_DIR / "fixed_b_ssm.csv", index=False)

ssm_metrics.to_csv(METRICS_DIR / "ssm_metrics.csv", index=False)
fit_summary.to_csv(METRICS_DIR / "ssm_fit_summary.csv", index=False)

print("Saved SSM prediction and metric files.")

## 5. 予測図の保存

テスト期間の実績値と SSM conditional forecast を比較する。学習期間の最後の24か月も薄く表示し、forecast の始点を確認しやすくする。

In [ ]:
def plot_split_forecast(split_name: str, save_path: Path) -> None:
    split = splits[split_name]
    plot_df = predictions_df[predictions_df["split"] == split_name]

    fig, ax = plt.subplots(figsize=(10.5, 5.5))
    train_tail = split["train"].tail(24)
    ax.plot(train_tail.index, train_tail["number_parcels"], color="0.55", linewidth=1.2, label="train actual tail")
    ax.plot(split["test"].index, split["test"]["number_parcels"], color="black", linewidth=1.6, label="test actual")
    ax.plot(plot_df["date"], plot_df["y_pred"], marker="o", linewidth=1.2, label="ssm conditional")

    ax.axvline(split["train"].index.max(), color="0.2", linestyle=":", linewidth=1.0)
    ax.set_title(f"{split_name}: SSM forecast comparison")
    ax.set_xlabel("Date")
    ax.set_ylabel("number_parcels")
    ax.grid(True, color="0.85", linewidth=0.8)
    ax.legend()
    fig.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


if "fixed_a" in splits:
    plot_split_forecast("fixed_a", FIGURES_DIR / "fixed_a_ssm_forecast.png")

if "fixed_b" in splits:
    plot_split_forecast("fixed_b", FIGURES_DIR / "fixed_b_ssm_forecast.png")

print("Saved SSM forecast figures.")

## 6. 読み取りメモ

この notebook では、主分析の状態空間モデルを予測評価基盤に接続できるかを確認した。評価は原系列の `number_parcels` で行うため、主分析の当てはまり指標とは直接同じ意味ではない。

比較では、seasonal naive / SARIMA / SARIMAX と同じ split・同じ共通フォーマットで SSM を見られるようになった。SSM は conditional forecast なので、test期間のイベントダミーを既知として与えている点に注意する。